# Comparative Study

In [1]:
import time
import torch
import torch.nn.functional as F
import torch.nn as nn
import numpy as np
import pandas as pd
from model_contrastive import Encoder, GRACE, drop_feature
from model import CNLinkPredictor, GCN
from NeighborOverlap import train, test
from ogbdataset import loaddataset, randomsplit
from ogb.linkproppred import PygLinkPropPredDataset, Evaluator
from torch.utils.tensorboard import SummaryWriter

from functools import partial
from torch.nn.functional import cosine_similarity
from utils_GCA import compute_pr, eigenvector_centrality
from torch.optim import AdamW
from torch_geometric.utils import dropout_adj, to_undirected, degree
from torch_geometric.data import Data
from torch_geometric.utils import to_dense_adj
from functional_GCA import drop_edge_weighted, pr_drop_weights, degree_drop_weights, evc_drop_weights, compute_pr, eigenvector_centrality, feature_drop_weights, drop_feature_weighted_2
from bgrl import BGRL, MLP_Predictor, GCN_BGRL,CosineDecayScheduler

In [2]:
hp = {
    'xdp': 0.7,
    'tdp': 0.3,
    'pt': 0.75,
    'gnnedp': 0.0,
    'preedp': 0.4,
    'predp': 0.05,
    'gnndp': 0.05,
    'probscale': 4.3,
    'proboffset': 2.8,
    'alpha': 1.0,
    'gnnlr': 0.0043,
    'prelr': 0.0024,
    'batch_size': 1152,
    'ln': True,
    'lnnn': True,
    'epochs': 100,
    'runs': 10,
    'hiddim': 256,
    'mplayers': 1,
    'testbs': 8192,
    'maskinput': True,
    'jk': True,
    'use_xlin': True,
    'tailact': True,
}
device = torch.device(f'cuda' if torch.cuda.is_available() else 'cpu')

## Generic function

In [3]:
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

In [4]:
def compute_table(res_dict):
    new_tab = [] 
    for key, result in res_dict.items():
        if isinstance(result, list):
            result = np.array(result)
            mean = round(100*np.mean(result), 2)
            std = round(100*np.std(result), 2)
            new_tab.append({
                'metric' : key,
                'mean' : mean,
                'std' : std
            })
    print(new_tab)
    df = pd.DataFrame(data=new_tab)
    res_latex = df.to_latex(index=False,
                  formatters={"name": str.upper},
                  float_format="{:.1f}".format)
    return new_tab, res_latex

In [5]:
for dataset in  ["Cora", "Citeseer", "Pubmed"]:
    print(f'################### {dataset} #################')
    if dataset in ["Cora", "Citeseer", "Pubmed"]:
        evaluator = Evaluator(name=f'ogbl-ppa')
        data, split_edge = loaddataset(dataset, False) # get a new split of dataset
        data = data.to(device)
        print(data)

################### Cora #################


/home/jovyan/envs/ncn/lib/python3.10/site-packages/torch_geometric/deprecation.py:12: UserWarning: 'train_test_split_edges' is deprecated, use 'transforms.RandomLinkSplit' instead
  warnings.warn(out)


2708 tensor(2707)
dataset split 
train edge 3696
valid edge 527
valid edge_neg 1055
test edge 1055
test edge_neg 1055
Data(x=[2708, 1433], edge_index=[2, 7392], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708], num_nodes=2708, adj_t=[2708, 2708, nnz=7392], max_x=-1, full_adj_t=[2708, 2708, nnz=7392])
################### Citeseer #################


/home/jovyan/envs/ncn/lib/python3.10/site-packages/torch_geometric/deprecation.py:12: UserWarning: 'train_test_split_edges' is deprecated, use 'transforms.RandomLinkSplit' instead
  warnings.warn(out)


3327 tensor(3326)
dataset split 
train edge 3187
valid edge 455
valid edge_neg 910
test edge 910
test edge_neg 910
Data(x=[3327, 3703], edge_index=[2, 6374], y=[3327], train_mask=[3327], val_mask=[3327], test_mask=[3327], num_nodes=3327, adj_t=[3327, 3327, nnz=6374], max_x=-1, full_adj_t=[3327, 3327, nnz=6374])
################### Pubmed #################


/home/jovyan/envs/ncn/lib/python3.10/site-packages/torch_geometric/deprecation.py:12: UserWarning: 'train_test_split_edges' is deprecated, use 'transforms.RandomLinkSplit' instead
  warnings.warn(out)


19717 tensor(19716)
dataset split 
train edge 31028
valid edge 4432
valid edge_neg 8864
test edge 8864
test edge_neg 8864
Data(x=[19717, 500], edge_index=[2, 62056], y=[19717], train_mask=[19717], val_mask=[19717], test_mask=[19717], num_nodes=19717, adj_t=[19717, 19717, nnz=62056], max_x=-1, full_adj_t=[19717, 19717, nnz=62056])


In [6]:
def legacy_train(epoch, model, predictor, data, split_edge, optimizer, evaluator, hp):
            t1 = time.time()
            loss = train(model, predictor, data, split_edge, optimizer,
                         hp['batch_size'], hp['maskinput'], [], None)
            if epoch % 10 == 0:
                print(f"10 train time {time.time()-t1:.2f} s, loss {loss:.4f}", flush=True)

In [7]:
def legacy_test(run, epoch, model, predictor, data, split_edge, evaluator, res_dict, writer, hp):
    t1 = time.time()
    results, h = test(model, predictor, data, split_edge, evaluator,
                   8192, False)
    print(f"test time {time.time()-t1:.2f} s")
    for key, result in results.items():
        writer.add_scalars(f"{key}_{run}", {
            "trn": result[0],
            "val": result[1],
            "tst": result[2]
        }, epoch)
        train_hits, valid_hits, test_hits = result
        res_dict[key].append(test_hits)
        print(key)
        print(f'Run: {run + 1:02d}, '
              f'Epoch: {epoch:02d}, '
              f'Train: {100 * train_hits:.2f}%, '
              f'Valid: {100 * valid_hits:.2f}%, '
              f'Test: {100 * test_hits:.2f}%')
    print('---', flush=True)
    return res_dict

In [8]:
class MlpProdDecoder(torch.nn.Module):
    """Hadamard-product-based MLP link predictor."""

    def __init__(self, embedding_size, hidden_size):
        super().__init__()
        self.embedding_size = embedding_size
        self.net = nn.Sequential(
            nn.Linear(embedding_size, hidden_size), nn.ReLU(), nn.Linear(hidden_size, 1)
        )

    def multidomainforward(self, x,
                           adj,
                           tar_ei, filled1: bool = False,
                           cndropprobs: list[float] = []):
        x1 = x[tar_ei[0]]
        x2 = x[tar_ei[1]]
        return self.net(x1 * x2)

    def forward(self, x, adj, tar_ei, filled1: bool = False):
        mdforward = self.multidomainforward(x, adj, tar_ei)
        return torch.cat([torch.sigmoid(mdforward)], dim=-1)

## contrastive pretrain

In [9]:
def pretrain_grace(model, data):
    param = {
        'learning_rate': 0.01,
        'num_hidden': 256,
        'num_proj_hidden': 32,
        'activation': 'prelu',
        'base_model': 'GCNConv',
        'num_layers': 2,
        'drop_edge_rate_1': 0.3,
        'drop_edge_rate_2': 0.4,
        'drop_feature_rate_1': 0.1,
        'drop_feature_rate_2': 0.0,
        'tau': 0.4,
        'num_epochs': 1500,
        'weight_decay': 1e-5,
        'drop_scheme': 'degree',
    }
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=param['learning_rate'],
        weight_decay=param['weight_decay']
    )
    t1 = time.time()
    for epoch in range(1, param['num_epochs'] + 1):
        model.train()
        optimizer.zero_grad()
        edge_index_1 = dropout_adj(data.edge_index, p=param[f'drop_edge_rate_{1}'])[0]
        edge_index_2 = dropout_adj(data.edge_index, p=param[f'drop_edge_rate_{2}'])[0]
        x_1 = drop_feature(data.x, param['drop_feature_rate_1'])
        x_2 = drop_feature(data.x, param['drop_feature_rate_2'])
        z1 = model(x_1, edge_index_1)
        z2 = model(x_2, edge_index_2)

        loss = model.loss(z1, z2)
        loss.backward()
        optimizer.step()
        if epoch % 100 == 0:
            print(f'(T) | Epoch={epoch:03d}, loss={loss:.4f}')
    print(f"pretrain time {time.time()-t1:.2f} s, loss {loss:.4f}", flush=True)

In [10]:
def pretrain_gca(model, data, drop_scheme='degree'):
    param = {
        'learning_rate': 0.01,
        'num_hidden': 256,
        'num_proj_hidden': 32,
        'activation': 'prelu',
        'base_model': 'GCNConv',
        'num_layers': 2,
        'drop_edge_rate_1': 0.3,
        'drop_edge_rate_2': 0.4,
        'drop_feature_rate_1': 0.1,
        'drop_feature_rate_2': 0.0,
        'tau': 0.4,
        'num_epochs': 1500,
        'weight_decay': 1e-5,
        'drop_scheme': drop_scheme,
    }
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=param['learning_rate'],
        weight_decay=param['weight_decay']
    )

    # compute drop_weights per centrality metrics
    if param['drop_scheme'] == 'degree':
        drop_weights = degree_drop_weights(data.edge_index).to(device)
    elif param['drop_scheme'] == 'pr':
        drop_weights = pr_drop_weights(data.edge_index, aggr='sink', k=200).to(device)
    elif param['drop_scheme'] == 'evc':
        drop_weights = evc_drop_weights(data).to(device)
    else:
        drop_weights = None

    # # compute feature_weights per centrality metrics
    if param['drop_scheme'] == 'degree':
        edge_index_ = to_undirected(data.edge_index)
        node_deg = degree(edge_index_[1])
        feature_weights = feature_drop_weights(data.x, node_c=node_deg).to(device)
    elif param['drop_scheme'] == 'pr':
        node_pr = compute_pr(data.edge_index)
        print(node_pr.shape, data.x.shape)
        feature_weights = feature_drop_weights(data.x, node_c=node_pr).to(device)
    elif param['drop_scheme'] == 'evc':
        node_evc = eigenvector_centrality(data)
        feature_weights = feature_drop_weights(data.x, node_c=node_evc).to(device)
    else:
        feature_weights = torch.ones((data.x.size(1),)).to(device)
    
    t1 = time.time()
    for epoch in range(1, param['num_epochs'] + 1):
        model.train()
        optimizer.zero_grad()
        edge_index_1 = drop_edge_weighted(data.edge_index, drop_weights, p=param[f'drop_edge_rate_{1}'], threshold=0.7)
        edge_index_2 = drop_edge_weighted(data.edge_index, drop_weights, p=param[f'drop_edge_rate_{2}'], threshold=0.7)
        x_1 = drop_feature_weighted_2(data.x, feature_weights, param['drop_feature_rate_1'])
        x_2 = drop_feature_weighted_2(data.x, feature_weights, param['drop_feature_rate_2'])

        z1 = model(x_1, edge_index_1)
        z2 = model(x_2, edge_index_2)

        loss = model.loss(z1, z2)
        loss.backward()
        optimizer.step()
        if epoch % 100 == 0:
            print(f'(T) | Epoch={epoch:03d}, loss={loss:.4f}')
    print(f"pretrain time {time.time()-t1:.2f} s, loss {loss:.4f}", flush=True)

In [11]:
def pretrain_bgrl(model, data):
    param = {
        'learning_rate': 0.01,
        'num_hidden': 256,
        'num_proj_hidden': 32,
        'activation': 'prelu',
        'base_model': 'GCNConv',
        'num_layers': 2,
        'drop_edge_rate_1': 0.3,
        'drop_edge_rate_2': 0.4,
        'drop_feature_rate_1': 0.1,
        'drop_feature_rate_2': 0.0,
        'tau': 0.4,
        'num_epochs': 1500,
        'weight_decay': 1e-5,
        'drop_scheme': 'degree',
    }
      # optimizer
    optimizer = AdamW(model.trainable_parameters(), lr=param['learning_rate'], weight_decay=param['weight_decay'])

    # scheduler
    lr_scheduler = CosineDecayScheduler(param['learning_rate'], 1000, param['num_epochs'])
    mm_scheduler = CosineDecayScheduler(1 - 0.99, 0, param['num_epochs'])

    t1 = time.time()
    for epoch in range(1, param['num_epochs'] + 1):
        model.train()

        lr = lr_scheduler.get(epoch)
        mm = 1 - mm_scheduler.get(epoch)


        optimizer.zero_grad()
        data_c1 = data.clone()
        data_c2 = data.clone()
        data_c1.edge_index = dropout_adj(data.edge_index, p=param[f'drop_edge_rate_{1}'])[0]
        data_c2.edge_index = dropout_adj(data.edge_index, p=param[f'drop_edge_rate_{2}'])[0]
        
        data_c1.x = drop_feature(data.x, param['drop_feature_rate_1'])
        data_c2.x = drop_feature(data.x, param['drop_feature_rate_2'])

        z1, y2 = model.train_forward(data_c1, data_c2)
        z2, y1 = model.train_forward(data_c2, data_c1)

        loss = 2 - cosine_similarity(z1, y2.detach(), dim=-1).mean() - cosine_similarity(z2, y1.detach(), dim=-1).mean() # loss simple
        loss.backward()
        optimizer.step()
        model.update_target_network(mm)
        if epoch % 100 == 0:
            print(f'(T) | Epoch={epoch:03d}, loss={loss:.4f}')
    print(f"pretrain time {time.time()-t1:.2f} s, loss {loss:.4f}", flush=True)

## Runs

In [12]:
def run(r, model, pretrain_function, predictor, data, evaluator, hp, input_dict):
    writer = SummaryWriter(f"./rec/GRACE_NCN")
    writer.add_text("hyperparams", str(hp)) 

    if pretrain_function is not None:
        pretrain_function(model, data)
    optimizer = torch.optim.Adam([{'params': model.parameters(), "lr": hp['gnnlr']}, 
       {'params': predictor.parameters(), 'lr': hp['prelr']}])

    for epoch in range(1, 1 + hp['epochs']):
        legacy_train(epoch, model, predictor, data, split_edge, optimizer, evaluator, hp)
    return legacy_test(r, epoch, model, predictor, data, split_edge, evaluator, input_dict, writer, hp)

### NCN vs MLP prod

In [11]:
for dataset in  ["Cora", "Citeseer", "Pubmed"]:
    print(f'################### {dataset} #################')
    if dataset in ["Cora", "Citeseer", "Pubmed"]:
        evaluator = Evaluator(name=f'ogbl-ppa')
        data, split_edge = loaddataset(dataset, False) # get a new split of dataset
        data = data.to(device)
    else:
        evaluator = Evaluator(name=f'ogbl-{args.dataset}')


    print('NCN base')
    res_dict = {"Hits@20": [], "Hits@20_std": 0, "Hits@50": [], "Hits@50_std": 0, "Hits@100": [], "Hits@100_std": 0}
    for r in range(hp['runs']):
        set_seed(r)
        model = GCN(data.num_features, hp['hiddim'], hp['hiddim'], hp['mplayers'],
                     hp['gnndp'], hp['ln'], False, data.max_x,
                     'puregcn', hp['jk'], hp['gnnedp'],  xdropout=hp['xdp'], taildropout=hp['tdp']).to(device)
        predictor = CNLinkPredictor(hp['hiddim'], hp['hiddim'], 1, 3,
                           hp['predp'], hp['preedp'], hp['lnnn']).to(device)
        res_dict = run(r, model, None, predictor, data, evaluator, hp, res_dict)
    res_dict_ncn_base, res_latex_ncn_base = compute_table(res_dict)
    print(res_dict_ncn_base)
    print('\n\n', res_latex_ncn_base, '\n\n')

    print('NCN MLPprod')
    res_dict = {"Hits@20": [], "Hits@20_std": 0, "Hits@50": [], "Hits@50_std": 0, "Hits@100": [], "Hits@100_std": 0}
    for r in range(hp['runs']):
        set_seed(r)
        model = GCN(data.num_features, hp['hiddim'], hp['hiddim'], hp['mplayers'],
                     hp['gnndp'], hp['ln'], False, data.max_x,
                     'puregcn', hp['jk'], hp['gnnedp'],  xdropout=hp['xdp'], taildropout=hp['tdp']).to(device)
        predictor2 = MlpProdDecoder(hp['hiddim'], hp['hiddim']).to(device)
        res_dict = run(r, model, None, predictor2, data, evaluator, hp, res_dict)
    res_dict_ncn_mlp_prod, res_latex_ncn_mlp_prod = compute_table(res_dict)
    print(res_dict_ncn_mlp_prod)
    print('\n\n', res_latex_ncn_mlp_prod, '\n\n')

################### Cora #################
2708 tensor(2707)
dataset split 
train edge 3696
valid edge 527
valid edge_neg 1055
test edge 1055
test edge_neg 1055
NCN base


/home/jovyan/envs/ncn/lib/python3.10/site-packages/torch_geometric/deprecation.py:12: UserWarning: 'train_test_split_edges' is deprecated, use 'transforms.RandomLinkSplit' instead
  warnings.warn(out)


10 train time 0.05 s, loss 1.2164
10 train time 0.05 s, loss 0.8786
10 train time 0.05 s, loss 0.7763
10 train time 0.04 s, loss 0.7137
10 train time 0.03 s, loss 0.6668
10 train time 0.05 s, loss 0.6556
10 train time 0.07 s, loss 0.6245
10 train time 0.06 s, loss 0.5939
10 train time 0.06 s, loss 0.5516
10 train time 0.08 s, loss 0.5483
test time 0.02 s
Hits@20
Run: 01, Epoch: 100, Train: 98.57%, Valid: 64.33%, Test: 69.29%
Hits@50
Run: 01, Epoch: 100, Train: 99.97%, Valid: 78.94%, Test: 82.27%
Hits@100
Run: 01, Epoch: 100, Train: 100.00%, Valid: 86.72%, Test: 90.14%
---
10 train time 0.06 s, loss 1.0756
10 train time 0.05 s, loss 0.7836
10 train time 0.05 s, loss 0.7163
10 train time 0.05 s, loss 0.6438
10 train time 0.04 s, loss 0.6257
10 train time 0.07 s, loss 0.6123
10 train time 0.07 s, loss 0.6128
10 train time 0.09 s, loss 0.5499
10 train time 0.06 s, loss 0.5841
10 train time 0.09 s, loss 0.5530
test time 0.01 s
Hits@20
Run: 02, Epoch: 100, Train: 98.86%, Valid: 65.09%, Test:

/tmp/ipykernel_11234/2585271375.py:15: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  res_latex = df.to_latex(index=False,


10 train time 0.01 s, loss 0.8677
10 train time 0.02 s, loss 0.7856
10 train time 0.01 s, loss 0.7029
10 train time 0.01 s, loss 0.6441
10 train time 0.01 s, loss 0.6554
10 train time 0.01 s, loss 0.6266
10 train time 0.02 s, loss 0.6025
10 train time 0.01 s, loss 0.5877
10 train time 0.02 s, loss 0.5877
test time 0.00 s
Hits@20
Run: 01, Epoch: 100, Train: 98.43%, Valid: 61.10%, Test: 63.13%
Hits@50
Run: 01, Epoch: 100, Train: 99.81%, Valid: 77.23%, Test: 80.09%
Hits@100
Run: 01, Epoch: 100, Train: 100.00%, Valid: 87.67%, Test: 88.44%
---
[{'metric': 'Hits@20', 'mean': 63.13, 'std': 0.0}, {'metric': 'Hits@50', 'mean': 80.09, 'std': 0.0}, {'metric': 'Hits@100', 'mean': 88.44, 'std': 0.0}]
10 train time 0.01 s, loss 0.9247


...

/tmp/ipykernel_11234/2585271375.py:15: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  res_latex = df.to_latex(index=False,


10 train time 0.28 s, loss 0.6161
10 train time 0.28 s, loss 0.5255
10 train time 0.30 s, loss 0.5019
10 train time 0.29 s, loss 0.4889
10 train time 0.22 s, loss 0.4847
10 train time 0.28 s, loss 0.4721
10 train time 0.23 s, loss 0.4589
10 train time 0.26 s, loss 0.4614
10 train time 0.30 s, loss 0.4562
10 train time 0.27 s, loss 0.4576
test time 0.01 s
Hits@20
Run: 09, Epoch: 100, Train: 34.06%, Valid: 32.49%, Test: 32.63%
Hits@50
Run: 09, Epoch: 100, Train: 57.57%, Valid: 51.67%, Test: 54.06%
Hits@100
Run: 09, Epoch: 100, Train: 76.97%, Valid: 66.74%, Test: 67.59%
---
[{'metric': 'Hits@20', 'mean': 38.17, 'std': 2.54}, {'metric': 'Hits@50', 'mean': 54.7, 'std': 1.13}, {'metric': 'Hits@100', 'mean': 68.78, 'std': 1.12}]


/tmp/ipykernel_11234/2585271375.py:15: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  res_latex = df.to_latex(index=False,


10 train time 0.29 s, loss 0.6059
10 train time 0.24 s, loss 0.5307
10 train time 0.25 s, loss 0.4962
10 train time 0.29 s, loss 0.4980
10 train time 0.22 s, loss 0.4756
10 train time 0.24 s, loss 0.4662
10 train time 0.26 s, loss 0.4539
10 train time 0.26 s, loss 0.4547
10 train time 0.26 s, loss 0.4529
10 train time 0.25 s, loss 0.4507
test time 0.01 s
Hits@20
Run: 10, Epoch: 100, Train: 34.25%, Valid: 31.70%, Test: 36.88%
Hits@50
Run: 10, Epoch: 100, Train: 55.71%, Valid: 49.95%, Test: 52.13%
Hits@100
Run: 10, Epoch: 100, Train: 77.82%, Valid: 66.79%, Test: 67.90%
---
[{'metric': 'Hits@20', 'mean': 38.04, 'std': 2.44}, {'metric': 'Hits@50', 'mean': 54.44, 'std': 1.32}, {'metric': 'Hits@100', 'mean': 68.69, 'std': 1.09}]
[{'metric': 'Hits@20', 'mean': 38.04, 'std': 2.44}, {'metric': 'Hits@50', 'mean': 54.44, 'std': 1.32}, {'metric': 'Hits@100', 'mean': 68.69, 'std': 1.09}]


 \begin{tabular}{lrr}
\toprule
  metric &  mean &  std \\
\midrule
 Hits@20 &  38.0 &  2.4 \\
 Hits@50 &  54.4

/tmp/ipykernel_11234/2585271375.py:15: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  res_latex = df.to_latex(index=False,


In [31]:
def run_legacy(r, dataset, evaluator, hp):
    print('NCN decoder')
    writer = SummaryWriter(f"./rec/GCN_NCN")
    writer.add_text("hyperparams", str(hp)) 
    if dataset in ["Cora", "Citeseer", "Pubmed"]:
        data, split_edge = loaddataset(dataset, False) # get a new split of dataset
        data = data.to(device)
    bestscore = None
    # build model
    model = GCN(data.num_features, hp['hiddim'], hp['hiddim'], hp['mplayers'],
                 hp['gnndp'], hp['ln'], False, data.max_x,
                 'puregcn', hp['jk'], hp['gnnedp'],  xdropout=hp['xdp'], taildropout=hp['tdp']).to(device)
    predictor = CNLinkPredictor(hp['hiddim'], hp['hiddim'], 1, 3,
                       hp['predp'], hp['preedp'], hp['lnnn']).to(device)
    optimizer = torch.optim.Adam([{'params': model.parameters(), "lr": hp['gnnlr']}, 
       {'params': predictor.parameters(), 'lr': hp['prelr']}])
    for epoch in range(1, 1 + hp['epochs']):
        legacy_train(epoch, model, predictor, data, split_edge, optimizer, evaluator, hp)
        if epoch % 100 == 0:
            legacy_test(r, epoch, model, predictor, data, split_edge, evaluator, bestscore, writer, hp)

In [23]:
for dataset in  ["Cora", "Citeseer", "Pubmed"]:
    print(f'################### {dataset} #################')
    if dataset in ["Cora", "Citeseer", "Pubmed"]:
        evaluator = Evaluator(name=f'ogbl-ppa')
    else:
        evaluator = Evaluator(name=f'ogbl-{args.dataset}')
    for r in range(hp['runs']):
        set_seed(r)
        run_mlp_prod_decoder(r, dataset, evaluator, hp)
        run_mlp_inner_prod_decoder(r, dataset, evaluator, hp)
        run_legacy(r, dataset, evaluator, hp)

################### Cora #################
2708 tensor(2707)
dataset split 
train edge 3696
valid edge 527
valid edge_neg 1055
test edge 1055
test edge_neg 1055


/home/jovyan/envs/ncn/lib/python3.10/site-packages/torch_geometric/deprecation.py:12: UserWarning: 'train_test_split_edges' is deprecated, use 'transforms.RandomLinkSplit' instead
  warnings.warn(out)


10 train time 0.02 s, loss 0.9147
10 train time 0.02 s, loss 0.7913
10 train time 0.02 s, loss 0.6992
10 train time 0.02 s, loss 0.6999
10 train time 0.05 s, loss 0.6370
10 train time 0.03 s, loss 0.6157
10 train time 0.02 s, loss 0.5965
10 train time 0.03 s, loss 0.5905
10 train time 0.02 s, loss 0.5835
10 train time 0.03 s, loss 0.5875
test time 0.00 s
Hits@20
Run: 01, Epoch: 100, Train: 99.40%, Valid: 71.35%, Test: 65.50%
Hits@50
Run: 01, Epoch: 100, Train: 99.95%, Valid: 82.35%, Test: 81.42%
Hits@100
Run: 01, Epoch: 100, Train: 100.00%, Valid: 90.32%, Test: 88.25%
---
################### Citeseer #################


/home/jovyan/envs/ncn/lib/python3.10/site-packages/torch_geometric/deprecation.py:12: UserWarning: 'train_test_split_edges' is deprecated, use 'transforms.RandomLinkSplit' instead
  warnings.warn(out)


3327 tensor(3325)
dataset split 
train edge 3187
valid edge 455
valid edge_neg 910
test edge 910
test edge_neg 910
10 train time 0.01 s, loss 0.9796
10 train time 0.01 s, loss 0.6308
10 train time 0.01 s, loss 0.5533
10 train time 0.01 s, loss 0.4645
10 train time 0.01 s, loss 0.4026
10 train time 0.02 s, loss 0.3940
10 train time 0.02 s, loss 0.3730
10 train time 0.02 s, loss 0.3303
10 train time 0.02 s, loss 0.3600
10 train time 0.02 s, loss 0.3348
test time 0.00 s
Hits@20
Run: 01, Epoch: 100, Train: 100.00%, Valid: 77.14%, Test: 76.70%
Hits@50
Run: 01, Epoch: 100, Train: 100.00%, Valid: 83.74%, Test: 85.27%
Hits@100
Run: 01, Epoch: 100, Train: 100.00%, Valid: 91.21%, Test: 88.79%
---
################### Pubmed #################


/home/jovyan/envs/ncn/lib/python3.10/site-packages/torch_geometric/deprecation.py:12: UserWarning: 'train_test_split_edges' is deprecated, use 'transforms.RandomLinkSplit' instead
  warnings.warn(out)


19717 tensor(19716)
dataset split 
train edge 31028
valid edge 4432
valid edge_neg 8864
test edge 8864
test edge_neg 8864
10 train time 0.25 s, loss 0.6862
10 train time 0.29 s, loss 0.5736
10 train time 0.27 s, loss 0.5278
10 train time 0.31 s, loss 0.5087
10 train time 0.26 s, loss 0.4971
10 train time 0.29 s, loss 0.4784
10 train time 0.23 s, loss 0.4735
10 train time 0.25 s, loss 0.4701
10 train time 0.25 s, loss 0.4548
10 train time 0.26 s, loss 0.4535
test time 0.01 s
Hits@20
Run: 01, Epoch: 100, Train: 33.33%, Valid: 32.42%, Test: 35.07%
Hits@50
Run: 01, Epoch: 100, Train: 59.50%, Valid: 51.33%, Test: 53.77%
Hits@100
Run: 01, Epoch: 100, Train: 77.02%, Valid: 65.43%, Test: 68.08%
---


### Grace with NCN and MLP prod

In [42]:
for dataset in  ["Cora", "Citeseer", "Pubmed"]:
    if dataset in ["Cora", "Citeseer", "Pubmed"]:
        evaluator = Evaluator(name=f'ogbl-ppa')
        data, split_edge = loaddataset(dataset, False) # get a new split of dataset
        data = data.to(device)
    else:
        evaluator = Evaluator(name=f'ogbl-{args.dataset}')

    if dataset == "Pubmed":
        hp['runs'] = 2
    else:
        hp['runs'] = 10
    
    res_dict = {"Hits@20": [], "Hits@20_std": 0, "Hits@50": [], "Hits@50_std": 0, "Hits@100": [], "Hits@100_std": 0}
    for r in range(hp['runs']):
        set_seed(r)
        basic_encoder = Encoder(data.num_features, hp['hiddim'], nn.Identity()).to(device)
        model = GRACE(basic_encoder, hp['hiddim'], 32).to(device)
        predictor = CNLinkPredictor(hp['hiddim'], hp['hiddim'], 1, 3,
                           hp['predp'], hp['preedp'], hp['lnnn']).to(device)
        res_dict = run(r, model, pretrain_grace, predictor, data, evaluator, hp, res_dict)
    res_dict_grace_ncn, res_latex_grace_ncn = compute_table(res_dict)
    print(f'######\t{dataset}\tGRACE NCN\t######')
    print(res_dict_grace_ncn)
    print('\n\n', res_latex_grace_ncn, '\n\n')

    res_dict = {"Hits@20": [], "Hits@20_std": 0, "Hits@50": [], "Hits@50_std": 0, "Hits@100": [], "Hits@100_std": 0}
    for r in range(hp['runs']):
        set_seed(r)
        basic_encoder = Encoder(data.num_features, hp['hiddim'], nn.Identity()).to(device)
        model = GRACE(basic_encoder, hp['hiddim'], 32).to(device)
        predictor2 = MlpProdDecoder(hp['hiddim'], hp['hiddim']).to(device)
        res_dict = run(r, model, pretrain_grace, predictor2, data, evaluator, hp, res_dict)
    res_dict_grace_mlp, res_latex_grace_mlp = compute_table(res_dict)
    print(f'######\t{dataset}\tGRACE MLP\t######')
    print(res_dict_grace_mlp)
    print('\n\n', res_latex_grace_mlp, '\n\n')

/home/jovyan/envs/ncn/lib/python3.10/site-packages/torch_geometric/deprecation.py:12: UserWarning: 'train_test_split_edges' is deprecated, use 'transforms.RandomLinkSplit' instead
  warnings.warn(out)


2708 tensor(2707)
dataset split 
train edge 3696
valid edge 527
valid edge_neg 1055
test edge 1055
test edge_neg 1055


/home/jovyan/envs/ncn/lib/python3.10/site-packages/torch_geometric/deprecation.py:12: UserWarning: 'dropout_adj' is deprecated, use 'dropout_edge' instead
  warnings.warn(out)


(T) | Epoch=100, loss=6.8046
(T) | Epoch=200, loss=6.7579
(T) | Epoch=300, loss=6.7400
(T) | Epoch=400, loss=6.7381
(T) | Epoch=500, loss=6.7428
(T) | Epoch=600, loss=6.7438
(T) | Epoch=700, loss=6.7416
(T) | Epoch=800, loss=6.7442
(T) | Epoch=900, loss=6.7453
(T) | Epoch=1000, loss=6.7404
(T) | Epoch=1100, loss=6.7345
(T) | Epoch=1200, loss=6.7360
(T) | Epoch=1300, loss=6.7340
(T) | Epoch=1400, loss=6.7338
(T) | Epoch=1500, loss=6.7369
pretrain time 25.67 s, loss 6.7369
10 train time 0.09 s, loss 0.3007
10 train time 0.07 s, loss 0.1734
10 train time 0.09 s, loss 0.1537
10 train time 0.06 s, loss 0.1352
10 train time 0.07 s, loss 0.1244
10 train time 0.09 s, loss 0.1029
10 train time 0.08 s, loss 0.0955
10 train time 0.08 s, loss 0.0865
10 train time 0.10 s, loss 0.0813
10 train time 0.06 s, loss 0.0722
test time 0.02 s
Hits@20
Run: 01, Epoch: 100, Train: 99.95%, Valid: 58.63%, Test: 59.53%
Hits@50
Run: 01, Epoch: 100, Train: 99.97%, Valid: 73.06%, Test: 70.05%
Hits@100
Run: 01, Epoch

/tmp/ipykernel_12925/2585271375.py:15: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  res_latex = df.to_latex(index=False,
/home/jovyan/envs/ncn/lib/python3.10/site-packages/torch_geometric/deprecation.py:12: UserWarning: 'dropout_adj' is deprecated, use 'dropout_edge' instead
  warnings.warn(out)


...

/home/jovyan/envs/ncn/lib/python3.10/site-packages/torch_geometric/deprecation.py:12: UserWarning: 'dropout_adj' is deprecated, use 'dropout_edge' instead
  warnings.warn(out)


(T) | Epoch=100, loss=8.9319
(T) | Epoch=200, loss=8.8885
(T) | Epoch=300, loss=8.8723
(T) | Epoch=400, loss=8.8663
(T) | Epoch=500, loss=8.8572
(T) | Epoch=600, loss=8.8502
(T) | Epoch=700, loss=8.8829
(T) | Epoch=800, loss=8.8444
(T) | Epoch=900, loss=8.8632
(T) | Epoch=1000, loss=8.8362
(T) | Epoch=1100, loss=8.8363
(T) | Epoch=1200, loss=8.8336
(T) | Epoch=1300, loss=8.8489
(T) | Epoch=1400, loss=8.8398
(T) | Epoch=1500, loss=8.8338
pretrain time 421.00 s, loss 8.8338
10 train time 0.67 s, loss 0.2425
10 train time 0.79 s, loss 0.1901
10 train time 0.73 s, loss 0.1613
10 train time 0.68 s, loss 0.1442
10 train time 0.67 s, loss 0.1315
10 train time 0.70 s, loss 0.1266
10 train time 0.69 s, loss 0.1163
10 train time 0.74 s, loss 0.1071
10 train time 0.79 s, loss 0.1050
10 train time 0.63 s, loss 0.0946
test time 0.04 s
Hits@20
Run: 01, Epoch: 100, Train: 95.77%, Valid: 56.14%, Test: 54.03%
Hits@50
Run: 01, Epoch: 100, Train: 99.76%, Valid: 68.93%, Test: 67.71%
Hits@100
Run: 01, Epoc

/tmp/ipykernel_12925/2585271375.py:15: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  res_latex = df.to_latex(index=False,
/home/jovyan/envs/ncn/lib/python3.10/site-packages/torch_geometric/deprecation.py:12: UserWarning: 'dropout_adj' is deprecated, use 'dropout_edge' instead
  warnings.warn(out)


(T) | Epoch=100, loss=8.9319
(T) | Epoch=200, loss=8.8885
(T) | Epoch=300, loss=8.8723
(T) | Epoch=400, loss=8.8663
(T) | Epoch=500, loss=8.8572
(T) | Epoch=600, loss=8.8502
(T) | Epoch=700, loss=8.8829
(T) | Epoch=800, loss=8.8444
(T) | Epoch=900, loss=8.8632
(T) | Epoch=1000, loss=8.8362
(T) | Epoch=1100, loss=8.8363
(T) | Epoch=1200, loss=8.8336
(T) | Epoch=1300, loss=8.8489
(T) | Epoch=1400, loss=8.8399
(T) | Epoch=1500, loss=8.8338
pretrain time 420.77 s, loss 8.8338
10 train time 0.46 s, loss 0.2532
10 train time 0.47 s, loss 0.2049
10 train time 0.39 s, loss 0.1795
10 train time 0.38 s, loss 0.1605
10 train time 0.42 s, loss 0.1418
10 train time 0.40 s, loss 0.1351
10 train time 0.41 s, loss 0.1275
10 train time 0.42 s, loss 0.1283
10 train time 0.38 s, loss 0.1231
10 train time 0.45 s, loss 0.1117
test time 0.02 s
Hits@20
Run: 01, Epoch: 100, Train: 87.73%, Valid: 52.69%, Test: 41.64%
Hits@50
Run: 01, Epoch: 100, Train: 98.41%, Valid: 67.40%, Test: 65.15%
Hits@100
Run: 01, Epoc

/tmp/ipykernel_12925/2585271375.py:15: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  res_latex = df.to_latex(index=False,


## GCA_deg with NCN and MLP

In [17]:
for dataset in  ["Cora", "Citeseer", "Pubmed"]:
    if dataset in ["Cora", "Citeseer", "Pubmed"]:
        evaluator = Evaluator(name=f'ogbl-ppa')
        data, split_edge = loaddataset(dataset, False) # get a new split of dataset
        data = data.to(device)
    else:
        evaluator = Evaluator(name=f'ogbl-{args.dataset}')

    if dataset == "Pubmed":
        hp['runs'] = 2
    else:
        hp['runs'] = 10
    
    res_dict = {"Hits@20": [], "Hits@20_std": 0, "Hits@50": [], "Hits@50_std": 0, "Hits@100": [], "Hits@100_std": 0}
    for r in range(hp['runs']):
        set_seed(r)
        pretrain_gca_pr = partial(pretrain_gca, drop_scheme='pr')
        basic_encoder = Encoder(data.num_features, hp['hiddim'], nn.Identity()).to(device)
        model = GRACE(basic_encoder, hp['hiddim'], 32).to(device)
        predictor = CNLinkPredictor(hp['hiddim'], hp['hiddim'], 1, 3,
                           hp['predp'], hp['preedp'], hp['lnnn']).to(device)
        res_dict = run(r, model, pretrain_gca_pr, predictor, data, evaluator, hp, res_dict)
    res_dict_grace_ncn, res_latex_grace_ncn = compute_table(res_dict)
    print(f'######\t{dataset}\GCA PR NCN\t######')
    print(res_dict_grace_ncn)
    print('\n\n', res_latex_grace_ncn, '\n\n')

    res_dict = {"Hits@20": [], "Hits@20_std": 0, "Hits@50": [], "Hits@50_std": 0, "Hits@100": [], "Hits@100_std": 0}
    for r in range(hp['runs']):
        set_seed(r)
        pretrain_gca_pr = partial(pretrain_gca, drop_scheme='pr')
        basic_encoder = Encoder(data.num_features, hp['hiddim'], nn.Identity()).to(device)
        model = GRACE(basic_encoder, hp['hiddim'], 32).to(device)
        predictor2 = MlpProdDecoder(hp['hiddim'], hp['hiddim']).to(device)
        res_dict = run(r, model, pretrain_gca_pr, predictor2, data, evaluator, hp, res_dict)
    res_dict_grace_mlp, res_latex_grace_mlp = compute_table(res_dict)
    print(f'######\t{dataset}\GCA PR MLP\t######')
    print(res_dict_grace_mlp)
    print('\n\n', res_latex_grace_mlp, '\n\n')

/home/jovyan/envs/ncn/lib/python3.10/site-packages/torch_geometric/deprecation.py:12: UserWarning: 'train_test_split_edges' is deprecated, use 'transforms.RandomLinkSplit' instead
  warnings.warn(out)


2708 tensor(2707)
dataset split 
train edge 3696
valid edge 527
valid edge_neg 1055
test edge 1055
test edge_neg 1055
(T) | Epoch=100, loss=7.4542
(T) | Epoch=200, loss=7.4277
(T) | Epoch=300, loss=7.4186
(T) | Epoch=400, loss=7.3374
(T) | Epoch=500, loss=7.3421
(T) | Epoch=600, loss=7.3005
(T) | Epoch=700, loss=7.3326
(T) | Epoch=800, loss=7.2785
(T) | Epoch=900, loss=7.2681
(T) | Epoch=1000, loss=7.2675
(T) | Epoch=1100, loss=7.3064
(T) | Epoch=1200, loss=7.2280
(T) | Epoch=1300, loss=7.2575
(T) | Epoch=1400, loss=7.2654
(T) | Epoch=1500, loss=7.2720
pretrain time 26.56 s, loss 7.2720
10 train time 0.10 s, loss 0.3989
10 train time 0.13 s, loss 0.2800
10 train time 0.08 s, loss 0.2367
10 train time 0.09 s, loss 0.1993
10 train time 0.07 s, loss 0.1583
10 train time 0.08 s, loss 0.1630
10 train time 0.10 s, loss 0.1439
10 train time 0.07 s, loss 0.1147
10 train time 0.09 s, loss 0.1173
10 train time 0.08 s, loss 0.0988
test time 0.02 s
Hits@20
Run: 01, Epoch: 100, Train: 99.73%, Valid

/tmp/ipykernel_15608/2585271375.py:15: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  res_latex = df.to_latex(index=False,


(T) | Epoch=100, loss=7.4542
(T) | Epoch=200, loss=7.4277
(T) | Epoch=300, loss=7.4186
(T) | Epoch=400, loss=7.3374
(T) | Epoch=500, loss=7.3421
(T) | Epoch=600, loss=7.3008
(T) | Epoch=700, loss=7.3281
(T) | Epoch=800, loss=7.2744
(T) | Epoch=900, loss=7.2706
(T) | Epoch=1000, loss=7.2631
(T) | Epoch=1100, loss=7.3139
(T) | Epoch=1200, loss=7.2210
(T) | Epoch=1300, loss=7.2609
(T) | Epoch=1400, loss=7.2812
(T) | Epoch=1500, loss=7.2512
pretrain time 27.07 s, loss 7.2512
10 train time 0.03 s, loss 0.9101
10 train time 0.04 s, loss 0.4175
10 train time 0.03 s, loss 0.3773
10 train time 0.04 s, loss 0.3495
10 train time 0.03 s, loss 0.2698
10 train time 0.02 s, loss 0.3048
10 train time 0.02 s, loss 0.2632
10 train time 0.02 s, loss 0.2238
10 train time 0.02 s, loss 0.2126
10 train time 0.02 s, loss 0.2087
test time 0.01 s
Hits@20
Run: 01, Epoch: 100, Train: 99.35%, Valid: 59.96%, Test: 49.86%
Hits@50
Run: 01, Epoch: 100, Train: 100.00%, Valid: 71.16%, Test: 68.72%
Hits@100
Run: 01, Epoc

/tmp/ipykernel_15608/2585271375.py:15: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  res_latex = df.to_latex(index=False,
/home/jovyan/envs/ncn/lib/python3.10/site-packages/torch_geometric/deprecation.py:12: UserWarning: 'train_test_split_edges' is deprecated, use 'transforms.RandomLinkSplit' instead
  warnings.warn(out)


...

/tmp/ipykernel_15608/2585271375.py:15: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  res_latex = df.to_latex(index=False,


(T) | Epoch=100, loss=6.9368
(T) | Epoch=200, loss=6.9131
(T) | Epoch=300, loss=6.9079
(T) | Epoch=400, loss=6.9058
(T) | Epoch=500, loss=6.9046
(T) | Epoch=600, loss=6.9043
(T) | Epoch=700, loss=6.9049
(T) | Epoch=800, loss=6.9055
(T) | Epoch=900, loss=6.9037
(T) | Epoch=1000, loss=6.9025
(T) | Epoch=1100, loss=6.9016
(T) | Epoch=1200, loss=6.9031
(T) | Epoch=1300, loss=6.9021
(T) | Epoch=1400, loss=6.9021
(T) | Epoch=1500, loss=6.9022
pretrain time 30.01 s, loss 6.9022
10 train time 0.03 s, loss 0.2216
10 train time 0.03 s, loss 0.1153
10 train time 0.03 s, loss 0.0937
10 train time 0.03 s, loss 0.0727
10 train time 0.03 s, loss 0.0664
10 train time 0.03 s, loss 0.0631
10 train time 0.03 s, loss 0.0610
10 train time 0.03 s, loss 0.0566
10 train time 0.03 s, loss 0.0413
10 train time 0.05 s, loss 0.0498
test time 0.01 s
Hits@20
Run: 01, Epoch: 100, Train: 100.00%, Valid: 65.49%, Test: 62.97%
Hits@50
Run: 01, Epoch: 100, Train: 100.00%, Valid: 72.09%, Test: 69.89%
Hits@100
Run: 01, Epo

/tmp/ipykernel_15608/2585271375.py:15: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  res_latex = df.to_latex(index=False,
/home/jovyan/envs/ncn/lib/python3.10/site-packages/torch_geometric/deprecation.py:12: UserWarning: 'train_test_split_edges' is deprecated, use 'transforms.RandomLinkSplit' instead
  warnings.warn(out)


19717 tensor(19714)
dataset split 
train edge 31028
valid edge 4432
valid edge_neg 8864
test edge 8864
test edge_neg 8864


RuntimeError: size mismatch, got 500, 500x19717,19715

## GCA_pr with NCN and MLP

In [18]:
for dataset in  ["Cora", "Citeseer", "Pubmed"]:
    if dataset in ["Cora", "Citeseer", "Pubmed"]:
        evaluator = Evaluator(name=f'ogbl-ppa')
        data, split_edge = loaddataset(dataset, False) # get a new split of dataset
        data = data.to(device)
    else:
        evaluator = Evaluator(name=f'ogbl-{args.dataset}')

    if dataset == "Pubmed":
        hp['runs'] = 2
    else:
        hp['runs'] = 10
    
    res_dict = {"Hits@20": [], "Hits@20_std": 0, "Hits@50": [], "Hits@50_std": 0, "Hits@100": [], "Hits@100_std": 0}
    for r in range(hp['runs']):
        set_seed(r)
        pretrain_gca_deg = partial(pretrain_gca, drop_scheme='degree')
        basic_encoder = Encoder(data.num_features, hp['hiddim'], nn.Identity()).to(device)
        model = GRACE(basic_encoder, hp['hiddim'], 32).to(device)
        predictor = CNLinkPredictor(hp['hiddim'], hp['hiddim'], 1, 3,
                           hp['predp'], hp['preedp'], hp['lnnn']).to(device)
        res_dict = run(r, model, pretrain_gca_deg, predictor, data, evaluator, hp, res_dict)
    res_dict_grace_ncn, res_latex_grace_ncn = compute_table(res_dict)
    print(f'######\t{dataset}\GCA NCN\t######')
    print(res_dict_grace_ncn)
    print('\n\n', res_latex_grace_ncn, '\n\n')

    res_dict = {"Hits@20": [], "Hits@20_std": 0, "Hits@50": [], "Hits@50_std": 0, "Hits@100": [], "Hits@100_std": 0}
    for r in range(hp['runs']):
        set_seed(r)
        pretrain_gca_deg = partial(pretrain_gca, drop_scheme='degree')
        basic_encoder = Encoder(data.num_features, hp['hiddim'], nn.Identity()).to(device)
        model = GRACE(basic_encoder, hp['hiddim'], 32).to(device)
        predictor2 = MlpProdDecoder(hp['hiddim'], hp['hiddim']).to(device)
        res_dict = run(r, model, pretrain_gca_deg, predictor2, data, evaluator, hp, res_dict)
    res_dict_grace_mlp, res_latex_grace_mlp = compute_table(res_dict)
    print(f'######\t{dataset}\GCA MLP\t######')
    print(res_dict_grace_mlp)
    print('\n\n', res_latex_grace_mlp, '\n\n')

/home/jovyan/envs/ncn/lib/python3.10/site-packages/torch_geometric/deprecation.py:12: UserWarning: 'train_test_split_edges' is deprecated, use 'transforms.RandomLinkSplit' instead
  warnings.warn(out)


2708 tensor(2707)
dataset split 
train edge 3696
valid edge 527
valid edge_neg 1055
test edge 1055
test edge_neg 1055
(T) | Epoch=100, loss=7.4623
(T) | Epoch=200, loss=7.4197
(T) | Epoch=300, loss=7.4238
(T) | Epoch=400, loss=7.3442
(T) | Epoch=500, loss=7.3343
(T) | Epoch=600, loss=7.2964
(T) | Epoch=700, loss=7.3345
(T) | Epoch=800, loss=7.2932
(T) | Epoch=900, loss=7.2836
(T) | Epoch=1000, loss=7.2796
(T) | Epoch=1100, loss=7.3119
(T) | Epoch=1200, loss=7.2253
(T) | Epoch=1300, loss=7.2721
(T) | Epoch=1400, loss=7.2933
(T) | Epoch=1500, loss=7.2541
pretrain time 26.65 s, loss 7.2541
10 train time 0.09 s, loss 0.3862
10 train time 0.07 s, loss 0.2846
10 train time 0.08 s, loss 0.2253
10 train time 0.07 s, loss 0.2103
10 train time 0.09 s, loss 0.1781
10 train time 0.07 s, loss 0.1435
10 train time 0.07 s, loss 0.1423
10 train time 0.07 s, loss 0.1237
10 train time 0.08 s, loss 0.1227
10 train time 0.07 s, loss 0.1193
test time 0.02 s
Hits@20
Run: 01, Epoch: 100, Train: 99.97%, Valid

/tmp/ipykernel_15608/2585271375.py:15: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  res_latex = df.to_latex(index=False,


(T) | Epoch=100, loss=7.4623
(T) | Epoch=200, loss=7.4197
(T) | Epoch=300, loss=7.4238
(T) | Epoch=400, loss=7.3442
(T) | Epoch=500, loss=7.3343
(T) | Epoch=600, loss=7.2961
(T) | Epoch=700, loss=7.3296
(T) | Epoch=800, loss=7.2879
(T) | Epoch=900, loss=7.2866
(T) | Epoch=1000, loss=7.2695
(T) | Epoch=1100, loss=7.3142
(T) | Epoch=1200, loss=7.2208
(T) | Epoch=1300, loss=7.2702
(T) | Epoch=1400, loss=7.2935
(T) | Epoch=1500, loss=7.2581
pretrain time 26.38 s, loss 7.2581
10 train time 0.05 s, loss 0.9184
10 train time 0.03 s, loss 0.4302
10 train time 0.03 s, loss 0.3593
10 train time 0.03 s, loss 0.3308
10 train time 0.03 s, loss 0.3029
10 train time 0.04 s, loss 0.2877
10 train time 0.03 s, loss 0.2729
10 train time 0.03 s, loss 0.2251
10 train time 0.03 s, loss 0.2226
10 train time 0.02 s, loss 0.2233
test time 0.00 s
Hits@20
Run: 01, Epoch: 100, Train: 99.95%, Valid: 63.57%, Test: 59.05%
Hits@50
Run: 01, Epoch: 100, Train: 100.00%, Valid: 74.38%, Test: 69.76%
Hits@100
Run: 01, Epoc

/tmp/ipykernel_15608/2585271375.py:15: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  res_latex = df.to_latex(index=False,
/home/jovyan/envs/ncn/lib/python3.10/site-packages/torch_geometric/deprecation.py:12: UserWarning: 'train_test_split_edges' is deprecated, use 'transforms.RandomLinkSplit' instead
  warnings.warn(out)


...

/tmp/ipykernel_15608/2585271375.py:15: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  res_latex = df.to_latex(index=False,


(T) | Epoch=100, loss=6.9341
(T) | Epoch=200, loss=6.9089
(T) | Epoch=300, loss=6.9079
(T) | Epoch=400, loss=6.9053
(T) | Epoch=500, loss=6.9042
(T) | Epoch=600, loss=6.9042
(T) | Epoch=700, loss=6.9042
(T) | Epoch=800, loss=6.9041
(T) | Epoch=900, loss=6.9063
(T) | Epoch=1000, loss=6.9015
(T) | Epoch=1100, loss=6.9012
(T) | Epoch=1200, loss=6.9033
(T) | Epoch=1300, loss=6.9015
(T) | Epoch=1400, loss=6.9004
(T) | Epoch=1500, loss=6.9003
pretrain time 30.19 s, loss 6.9003
10 train time 0.03 s, loss 0.1847
10 train time 0.03 s, loss 0.1034
10 train time 0.04 s, loss 0.0828
10 train time 0.03 s, loss 0.0679
10 train time 0.03 s, loss 0.0553
10 train time 0.02 s, loss 0.0682
10 train time 0.02 s, loss 0.0626
10 train time 0.02 s, loss 0.0546
10 train time 0.02 s, loss 0.0576
10 train time 0.02 s, loss 0.0551
test time 0.01 s
Hits@20
Run: 01, Epoch: 100, Train: 100.00%, Valid: 63.74%, Test: 63.19%
Hits@50
Run: 01, Epoch: 100, Train: 100.00%, Valid: 73.41%, Test: 68.90%
Hits@100
Run: 01, Epo

/tmp/ipykernel_15608/2585271375.py:15: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  res_latex = df.to_latex(index=False,
/home/jovyan/envs/ncn/lib/python3.10/site-packages/torch_geometric/deprecation.py:12: UserWarning: 'train_test_split_edges' is deprecated, use 'transforms.RandomLinkSplit' instead
  warnings.warn(out)


19717 tensor(19714)
dataset split 
train edge 31028
valid edge 4432
valid edge_neg 8864
test edge 8864
test edge_neg 8864


RuntimeError: size mismatch, got 500, 500x19717,19715

## GCA_eigenvalues with NCN and MLP

In [13]:
for dataset in  ["Cora", "Citeseer", "Pubmed"]:
    if dataset in ["Cora", "Citeseer", "Pubmed"]:
        evaluator = Evaluator(name=f'ogbl-ppa')
        data, split_edge = loaddataset(dataset, False) # get a new split of dataset
        data = data.to(device)
    else:
        evaluator = Evaluator(name=f'ogbl-{args.dataset}')

    if dataset == "Pubmed":
        hp['runs'] = 2
    else:
        hp['runs'] = 10
    
    res_dict = {"Hits@20": [], "Hits@20_std": 0, "Hits@50": [], "Hits@50_std": 0, "Hits@100": [], "Hits@100_std": 0}
    for r in range(hp['runs']):
        set_seed(r)
        pretrain_gca_egv = partial(pretrain_gca, drop_scheme='evc')
        basic_encoder = Encoder(data.num_features, hp['hiddim'], nn.Identity()).to(device)
        model = GRACE(basic_encoder, hp['hiddim'], 32).to(device)
        predictor = CNLinkPredictor(hp['hiddim'], hp['hiddim'], 1, 3,
                           hp['predp'], hp['preedp'], hp['lnnn']).to(device)
        res_dict = run(r, model, pretrain_gca_egv, predictor, data, evaluator, hp, res_dict)
    res_dict_grace_ncn, res_latex_grace_ncn = compute_table(res_dict)
    print(f'######\t{dataset}\GCA egv NCN\t######')
    print(res_dict_grace_ncn)
    print('\n\n', res_latex_grace_ncn, '\n\n')

    res_dict = {"Hits@20": [], "Hits@20_std": 0, "Hits@50": [], "Hits@50_std": 0, "Hits@100": [], "Hits@100_std": 0}
    for r in range(hp['runs']):
        set_seed(r)
        pretrain_gca_egv = partial(pretrain_gca, drop_scheme='evc')
        basic_encoder = Encoder(data.num_features, hp['hiddim'], nn.Identity()).to(device)
        model = GRACE(basic_encoder, hp['hiddim'], 32).to(device)
        predictor2 = MlpProdDecoder(hp['hiddim'], hp['hiddim']).to(device)
        res_dict = run(r, model, pretrain_gca_egv, predictor2, data, evaluator, hp, res_dict)
    res_dict_grace_mlp, res_latex_grace_mlp = compute_table(res_dict)
    print(f'######\t{dataset}\GCA egv MLP\t######')
    print(res_dict_grace_mlp)
    print('\n\n', res_latex_grace_mlp, '\n\n')

/home/jovyan/envs/ncn/lib/python3.10/site-packages/torch_geometric/deprecation.py:12: UserWarning: 'train_test_split_edges' is deprecated, use 'transforms.RandomLinkSplit' instead
  warnings.warn(out)


2708 tensor(2707)
dataset split 
train edge 3696
valid edge 527
valid edge_neg 1055
test edge 1055
test edge_neg 1055
(T) | Epoch=100, loss=7.4854
(T) | Epoch=200, loss=7.4143
(T) | Epoch=300, loss=7.3786
(T) | Epoch=400, loss=7.3396
(T) | Epoch=500, loss=7.3422
(T) | Epoch=600, loss=7.2888
(T) | Epoch=700, loss=7.3210
(T) | Epoch=800, loss=7.2562
(T) | Epoch=900, loss=7.2758
(T) | Epoch=1000, loss=7.2356
(T) | Epoch=1100, loss=7.2500
(T) | Epoch=1200, loss=7.2094
(T) | Epoch=1300, loss=7.2455
(T) | Epoch=1400, loss=7.2466
(T) | Epoch=1500, loss=7.2142
pretrain time 30.50 s, loss 7.2142
10 train time 0.09 s, loss 0.3964
10 train time 0.11 s, loss 0.2731
10 train time 0.11 s, loss 0.2205
10 train time 0.10 s, loss 0.1966
10 train time 0.13 s, loss 0.1500
10 train time 0.11 s, loss 0.1442
10 train time 0.09 s, loss 0.1232
10 train time 0.10 s, loss 0.1154
10 train time 0.15 s, loss 0.1147
10 train time 0.11 s, loss 0.1061
test time 0.03 s
Hits@20
Run: 01, Epoch: 100, Train: 100.00%, Vali

/tmp/ipykernel_18807/2585271375.py:15: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  res_latex = df.to_latex(index=False,


(T) | Epoch=100, loss=7.4854
(T) | Epoch=200, loss=7.4143
(T) | Epoch=300, loss=7.3786
(T) | Epoch=400, loss=7.3396
(T) | Epoch=500, loss=7.3422
(T) | Epoch=600, loss=7.2888
(T) | Epoch=700, loss=7.3217
(T) | Epoch=800, loss=7.2538
(T) | Epoch=900, loss=7.2805
(T) | Epoch=1000, loss=7.2397
(T) | Epoch=1100, loss=7.2531
(T) | Epoch=1200, loss=7.2016
(T) | Epoch=1300, loss=7.2406
(T) | Epoch=1400, loss=7.2442
(T) | Epoch=1500, loss=7.2199
pretrain time 26.71 s, loss 7.2199
10 train time 0.04 s, loss 0.8162
10 train time 0.03 s, loss 0.4004
10 train time 0.06 s, loss 0.3444
10 train time 0.04 s, loss 0.2897
10 train time 0.03 s, loss 0.2921
10 train time 0.03 s, loss 0.2722
10 train time 0.03 s, loss 0.2224
10 train time 0.03 s, loss 0.2122
10 train time 0.02 s, loss 0.2162
10 train time 0.02 s, loss 0.1853
test time 0.01 s
Hits@20
Run: 01, Epoch: 100, Train: 99.19%, Valid: 55.98%, Test: 62.56%
Hits@50
Run: 01, Epoch: 100, Train: 100.00%, Valid: 72.87%, Test: 73.74%
Hits@100
Run: 01, Epoc

/tmp/ipykernel_18807/2585271375.py:15: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  res_latex = df.to_latex(index=False,
/home/jovyan/envs/ncn/lib/python3.10/site-packages/torch_geometric/deprecation.py:12: UserWarning: 'train_test_split_edges' is deprecated, use 'transforms.RandomLinkSplit' instead
  warnings.warn(out)


...

/tmp/ipykernel_18807/2585271375.py:15: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  res_latex = df.to_latex(index=False,
/home/jovyan/envs/ncn/lib/python3.10/site-packages/torch_geometric/deprecation.py:12: UserWarning: 'train_test_split_edges' is deprecated, use 'transforms.RandomLinkSplit' instead
  warnings.warn(out)


19717 tensor(19714)
dataset split 
train edge 31028
valid edge 4432
valid edge_neg 8864
test edge 8864
test edge_neg 8864
(T) | Epoch=100, loss=8.9562
(T) | Epoch=200, loss=8.8980
(T) | Epoch=300, loss=8.9148
(T) | Epoch=400, loss=8.9076
(T) | Epoch=500, loss=8.8713
(T) | Epoch=600, loss=8.8766
(T) | Epoch=700, loss=8.8568
(T) | Epoch=800, loss=8.8750
(T) | Epoch=900, loss=8.8625
(T) | Epoch=1000, loss=8.8563
(T) | Epoch=1100, loss=8.8474
(T) | Epoch=1200, loss=8.8874
(T) | Epoch=1300, loss=8.8565
(T) | Epoch=1400, loss=8.8475
(T) | Epoch=1500, loss=8.8632
pretrain time 468.41 s, loss 8.8632
10 train time 0.82 s, loss 0.2331
10 train time 0.71 s, loss 0.1810
10 train time 0.78 s, loss 0.1706
10 train time 0.75 s, loss 0.1464
10 train time 1.01 s, loss 0.1339
10 train time 0.95 s, loss 0.1311
10 train time 0.92 s, loss 0.1153
10 train time 1.07 s, loss 0.1083
10 train time 0.85 s, loss 0.1055
10 train time 0.73 s, loss 0.0997
test time 0.04 s
Hits@20
Run: 01, Epoch: 100, Train: 98.21%, 

/tmp/ipykernel_18807/2585271375.py:15: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  res_latex = df.to_latex(index=False,


(T) | Epoch=100, loss=8.9562
(T) | Epoch=200, loss=8.8980
(T) | Epoch=300, loss=8.9148
(T) | Epoch=400, loss=8.9076
(T) | Epoch=500, loss=8.8713
(T) | Epoch=600, loss=8.8766
(T) | Epoch=700, loss=8.8568
(T) | Epoch=800, loss=8.8750
(T) | Epoch=900, loss=8.8625
(T) | Epoch=1000, loss=8.8563
(T) | Epoch=1100, loss=8.8475
(T) | Epoch=1200, loss=8.8874
(T) | Epoch=1300, loss=8.8565
(T) | Epoch=1400, loss=8.8475
(T) | Epoch=1500, loss=8.8633
pretrain time 463.70 s, loss 8.8633
10 train time 0.46 s, loss 0.2489
10 train time 0.38 s, loss 0.1927
10 train time 0.41 s, loss 0.1703
10 train time 0.47 s, loss 0.1498
10 train time 0.45 s, loss 0.1392
10 train time 0.46 s, loss 0.1348
10 train time 0.46 s, loss 0.1296
10 train time 0.51 s, loss 0.1279
10 train time 0.59 s, loss 0.1214
10 train time 0.60 s, loss 0.1171
test time 0.02 s
Hits@20
Run: 01, Epoch: 100, Train: 79.49%, Valid: 46.55%, Test: 48.63%
Hits@50
Run: 01, Epoch: 100, Train: 97.58%, Valid: 65.84%, Test: 64.80%
Hits@100
Run: 01, Epoc

/tmp/ipykernel_18807/2585271375.py:15: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  res_latex = df.to_latex(index=False,


## BGRL with NCN and MLP prod

In [13]:
for dataset in  ["Cora", "Citeseer", "Pubmed"]:
    if dataset in ["Cora", "Citeseer", "Pubmed"]:
        evaluator = Evaluator(name=f'ogbl-ppa')
        data, split_edge = loaddataset(dataset, False) # get a new split of dataset
        data = data.to(device)
    else:
        evaluator = Evaluator(name=f'ogbl-{args.dataset}')

    if dataset == "Pubmed":
        hp['runs'] = 2
    else:
        hp["runs"] = 10
    
    res_dict = {"Hits@20": [], "Hits@20_std": 0, "Hits@50": [], "Hits@50_std": 0, "Hits@100": [], "Hits@100_std": 0}
    for r in range(hp['runs']):
        set_seed(r)
        basic_encoder = GCN_BGRL([data.num_features, hp['hiddim']], batchnorm=True).to(device)
        predictor = MLP_Predictor(hp['hiddim'], hp['hiddim']).to(device)
        model = BGRL(basic_encoder, predictor).to(device)
        predictor = CNLinkPredictor(hp['hiddim'], hp['hiddim'], 1, 3,
                           hp['predp'], hp['preedp'], hp['lnnn']).to(device)
        res_dict = run(r, model, pretrain_bgrl, predictor, data, evaluator, hp, res_dict)
    res_dict_bgrl_ncn, res_latex_bgrl_ncn = compute_table(res_dict)
    print(f'######\t{dataset}\t BGRL NCN\t######')
    print(res_dict_bgrl_ncn)
    print('\n\n', res_latex_bgrl_ncn, '\n\n')

    res_dict = {"Hits@20": [], "Hits@20_std": 0, "Hits@50": [], "Hits@50_std": 0, "Hits@100": [], "Hits@100_std": 0}
    for r in range(hp['runs']):
        set_seed(r)
        basic_encoder = GCN_BGRL([data.num_features, hp['hiddim']], batchnorm=True).to(device)
        predictor = MLP_Predictor(hp['hiddim'], hp['hiddim']).to(device)
        model = BGRL(basic_encoder, predictor).to(device)
        predictor2 = MlpProdDecoder(hp['hiddim'], hp['hiddim']).to(device)
        res_dict = run(r, model, pretrain_bgrl, predictor2, data, evaluator, hp, res_dict)
    res_dict_bgrl_mlp, res_latex_bgrl_mlp = compute_table(res_dict)
    print(f'######\t{dataset}\t BGRL MLP\t######')
    print(res_dict_bgrl_mlp)
    print('\n\n', res_latex_bgrl_mlp, '\n\n')

/home/jovyan/envs/ncn/lib/python3.10/site-packages/torch_geometric/deprecation.py:12: UserWarning: 'train_test_split_edges' is deprecated, use 'transforms.RandomLinkSplit' instead
  warnings.warn(out)


2708 tensor(2707)
dataset split 
train edge 3696
valid edge 527
valid edge_neg 1055
test edge 1055
test edge_neg 1055


/home/jovyan/envs/ncn/lib/python3.10/site-packages/torch_geometric/deprecation.py:12: UserWarning: 'dropout_adj' is deprecated, use 'dropout_edge' instead
  warnings.warn(out)


(T) | Epoch=100, loss=0.2746
(T) | Epoch=200, loss=0.1922
(T) | Epoch=300, loss=0.1730
(T) | Epoch=400, loss=0.1486
(T) | Epoch=500, loss=0.1384
(T) | Epoch=600, loss=0.1294
(T) | Epoch=700, loss=0.1274
(T) | Epoch=800, loss=0.1224
(T) | Epoch=900, loss=0.1175
(T) | Epoch=1000, loss=0.1154
(T) | Epoch=1100, loss=0.1141
(T) | Epoch=1200, loss=0.1055
(T) | Epoch=1300, loss=0.1105
(T) | Epoch=1400, loss=0.1058
(T) | Epoch=1500, loss=0.1125
pretrain time 19.11 s, loss 0.1125
10 train time 0.08 s, loss 0.0968
10 train time 0.06 s, loss 0.0561
10 train time 0.06 s, loss 0.0562
10 train time 0.08 s, loss 0.0454
10 train time 0.08 s, loss 0.0434
10 train time 0.07 s, loss 0.0528
10 train time 0.06 s, loss 0.0489
10 train time 0.09 s, loss 0.0436
10 train time 0.07 s, loss 0.0375
10 train time 0.06 s, loss 0.0493
test time 0.02 s
Hits@20
Run: 01, Epoch: 100, Train: 99.89%, Valid: 59.58%, Test: 57.25%
Hits@50
Run: 01, Epoch: 100, Train: 99.89%, Valid: 72.11%, Test: 73.93%
Hits@100
Run: 01, Epoch

/tmp/ipykernel_15608/2585271375.py:15: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  res_latex = df.to_latex(index=False,
/home/jovyan/envs/ncn/lib/python3.10/site-packages/torch_geometric/deprecation.py:12: UserWarning: 'dropout_adj' is deprecated, use 'dropout_edge' instead
  warnings.warn(out)


...

/home/jovyan/envs/ncn/lib/python3.10/site-packages/torch_geometric/deprecation.py:12: UserWarning: 'dropout_adj' is deprecated, use 'dropout_edge' instead
  warnings.warn(out)


(T) | Epoch=100, loss=0.2595
(T) | Epoch=200, loss=0.2468
(T) | Epoch=300, loss=0.2374
(T) | Epoch=400, loss=0.2173
(T) | Epoch=500, loss=0.2084
(T) | Epoch=600, loss=0.1895
(T) | Epoch=700, loss=0.2021
(T) | Epoch=800, loss=0.1994
(T) | Epoch=900, loss=0.2025
(T) | Epoch=1000, loss=0.2006
(T) | Epoch=1100, loss=0.1950
(T) | Epoch=1200, loss=0.1858
(T) | Epoch=1300, loss=0.1981
(T) | Epoch=1400, loss=0.1785
(T) | Epoch=1500, loss=0.1780
pretrain time 36.25 s, loss 0.1780
10 train time 0.64 s, loss 0.1364
10 train time 0.61 s, loss 0.1049
10 train time 0.59 s, loss 0.0871
10 train time 0.56 s, loss 0.0819
10 train time 0.61 s, loss 0.0685
10 train time 0.62 s, loss 0.0674
10 train time 0.62 s, loss 0.0595
10 train time 0.56 s, loss 0.0580
10 train time 0.53 s, loss 0.0580
10 train time 0.64 s, loss 0.0574
test time 0.04 s
Hits@20
Run: 01, Epoch: 100, Train: 98.54%, Valid: 53.50%, Test: 53.36%
Hits@50
Run: 01, Epoch: 100, Train: 99.93%, Valid: 71.62%, Test: 68.73%
Hits@100
Run: 01, Epoch

/tmp/ipykernel_15608/2585271375.py:15: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  res_latex = df.to_latex(index=False,
/home/jovyan/envs/ncn/lib/python3.10/site-packages/torch_geometric/deprecation.py:12: UserWarning: 'dropout_adj' is deprecated, use 'dropout_edge' instead
  warnings.warn(out)


(T) | Epoch=100, loss=0.2611
(T) | Epoch=200, loss=0.2556
(T) | Epoch=300, loss=0.3532
(T) | Epoch=400, loss=0.2113
(T) | Epoch=500, loss=0.2067
(T) | Epoch=600, loss=0.1912
(T) | Epoch=700, loss=0.2030
(T) | Epoch=800, loss=0.2008
(T) | Epoch=900, loss=0.2030
(T) | Epoch=1000, loss=0.2015
(T) | Epoch=1100, loss=0.1958
(T) | Epoch=1200, loss=0.1865
(T) | Epoch=1300, loss=0.1988
(T) | Epoch=1400, loss=0.1792
(T) | Epoch=1500, loss=0.1792
pretrain time 36.04 s, loss 0.1792
10 train time 0.37 s, loss 0.1463
10 train time 0.28 s, loss 0.1050
10 train time 0.28 s, loss 0.0951
10 train time 0.30 s, loss 0.0805
10 train time 0.34 s, loss 0.0729
10 train time 0.36 s, loss 0.0653
10 train time 0.31 s, loss 0.0628
10 train time 0.36 s, loss 0.0603
10 train time 0.29 s, loss 0.0584
10 train time 0.38 s, loss 0.0539
test time 0.02 s
Hits@20
Run: 01, Epoch: 100, Train: 98.30%, Valid: 53.79%, Test: 51.24%
Hits@50
Run: 01, Epoch: 100, Train: 99.95%, Valid: 69.18%, Test: 65.67%
Hits@100
Run: 01, Epoch

/tmp/ipykernel_15608/2585271375.py:15: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  res_latex = df.to_latex(index=False,
